# Native BioT5 Diverse Beam Review On Kaggle

This notebook clones the repo, ensures ChEBI-20 exists locally, runs a small native BioT5 text2mol review on selected ChEBI descriptions, and saves raw and parsed review outputs for download from the Kaggle working directory.
It mirrors the maintained SELFIES-first decoding and diverse-beam review style used by the local review notebooks.


## Notes

- Enable internet so Kaggle can clone the repo, fetch the BioT5 checkpoint, and load the diverse beam backend if needed.
- Enable a GPU accelerator because BioT5 generation is expensive on CPU.
- Add an optional Kaggle secret named `HF_TOKEN` if you want authenticated Hugging Face access.
- This notebook uses the official `QizhiPei/biot5-plus-base-chebi20` tokenizer and the shared SELFIES-first review helpers from the repo.


In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/kaggle/working/Thesis")
STAGE_NAME = "review_biot5_diverse_beam"

%cd /kaggle/working
!if [ -d "{REPO_DIR / '.git'}" ]; then echo "Reusing {REPO_DIR}"; elif [ -d "{REPO_DIR}" ]; then echo "Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook." && false; else git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" fetch --depth 1 origin "{REPO_BRANCH}" && git -C "{REPO_DIR}" checkout -B "{REPO_BRANCH}" FETCH_HEAD

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from thesis_kaggle_support import (
    ensure_runtime_dependencies,
    export_stage_artifacts,
    import_or_none,
    json_dumps,
    pip_install,
    report_runtime,
    write_jsonl,
)


In [ ]:
CHEBI_OUTPUT_DIR = REPO_DIR / "data" / "chebi20"
CHEBI_PROCESSED_DIR = CHEBI_OUTPUT_DIR / "processed"
OUTPUT_DIR = REPO_DIR / "outputs" / "kaggle" / STAGE_NAME
RAW_GENERATIONS_PATH = OUTPUT_DIR / "raw_generations.jsonl"
REVIEW_ROWS_PATH = OUTPUT_DIR / "review_rows.jsonl"
SUMMARY_ROWS_PATH = OUTPUT_DIR / "summary_rows.json"
CONFIG_SNAPSHOT_PATH = OUTPUT_DIR / "config_snapshot.json"

ensure_runtime_dependencies(REPO_DIR)
if import_or_none("huggingface_hub") is None:
    pip_install("huggingface_hub")
if import_or_none("sentencepiece") is None:
    pip_install("sentencepiece")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_login_status = "not_attempted"
hf_token = None
try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    hf_login_status = f"secret_unavailable:{exc.__class__.__name__}"

if hf_token:
    login(token=hf_token)
    hf_login_status = "logged_in"
elif hf_login_status == "not_attempted":
    hf_login_status = "no_hf_token"

runtime_report = report_runtime(require_gpu=True)

print(json_dumps({
    "runtime": runtime_report,
    "hf_login_status": hf_login_status,
    "repo_dir": str(REPO_DIR),
    "chebi_processed_dir": str(CHEBI_PROCESSED_DIR),
    "output_dir": str(OUTPUT_DIR),
}))


In [ ]:
have_processed = all((CHEBI_PROCESSED_DIR / f"{split}.jsonl").exists() for split in ("train", "validation", "test"))
print(json_dumps({
    "have_processed": have_processed,
    "processed_dir": str(CHEBI_PROCESSED_DIR),
}))

%cd {REPO_DIR}
!if [ -f "{CHEBI_PROCESSED_DIR / 'train.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'validation.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'test.jsonl'}" ]; then echo "ChEBI processed splits already exist"; else python scripts/download_chebi20.py --output-dir "{CHEBI_OUTPUT_DIR}"; fi


In [ ]:
from __future__ import annotations

import inspect
import json
import time

import selfies
import torch
import transformers
from transformers import T5ForConditionalGeneration, T5Tokenizer

from molecules.collection.filtering import CollectionMetricConfig, prepare_reference_groups
from notebooks.biot5_collection_review_support import (
    assess_biot5_native_generation_output,
    summarize_biot5_native_review_records,
)
from src.io_utils import read_jsonl
from src.prompting import build_text2mol_prompt
from src.training import choose_device


def print_section(title: str) -> None:
    print(f"\n=== {title} ===")


def print_json(title: str, payload) -> None:
    print_section(title)
    print(json.dumps(payload, indent=2, ensure_ascii=False))


def print_records(title: str, records, *, limit: int | None = None, keys: list[str] | None = None) -> None:
    payload = list(records)
    total = len(payload)
    if keys is not None:
        payload = [{key: row.get(key) for key in keys} for row in payload]
    shown = payload if limit is None else payload[:limit]
    print_section(f"{title} (showing {len(shown)} of {total})")
    print(json.dumps(shown, indent=2, ensure_ascii=False))


version_report = {
    "python": str(__import__("sys").version.split()[0]),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "selfies": selfies.__version__,
    "repo_dir": str(REPO_DIR),
}
print_json("Environment", version_report)


In [ ]:
TRAIN_FILE = CHEBI_PROCESSED_DIR / "train.jsonl"
MODEL_NAME_OR_PATH = "QizhiPei/biot5-plus-base-chebi20"
DEVICE = "auto"

SELECTED_DESCRIPTION_IDS = ["129626631", "12699"]
NUM_SAMPLES = 30
MODEL_MAX_LENGTH = 512
ALLOW_FILTER_FALLBACK = True

GENERATION_CONFIGS = {
    "greedy_native": {
        "target_count": 1,
        "max_length": 512,
        "num_beams": 1,
        "num_return_sequences": 1,
    },
    "diverse_beam_fast": {
        "target_count": NUM_SAMPLES,
        "max_length": 512,
        "num_beams": 30,
        "num_return_sequences": 30,
        "num_beam_groups": 5,
        "diversity_penalty": 0.3,
        "early_stopping": True,
        "length_penalty": 1.0,
    },
    "diverse_beam_alt": {
        "target_count": NUM_SAMPLES,
        "max_length": 512,
        "num_beams": 30,
        "num_return_sequences": 30,
        "num_beam_groups": 6,
        "diversity_penalty": 0.5,
        "early_stopping": True,
        "length_penalty": 1.0,
    },
}
METRIC_CONFIG = CollectionMetricConfig(
    fingerprint_radius=2,
    fingerprint_num_bits=2048,
    acceptance_dice_threshold=0.7,
)

print_json("Notebook Config", {
    "train_file": str(TRAIN_FILE),
    "model_name_or_path": MODEL_NAME_OR_PATH,
    "device": DEVICE,
    "selected_description_ids": SELECTED_DESCRIPTION_IDS,
    "num_samples": NUM_SAMPLES,
    "model_max_length": MODEL_MAX_LENGTH,
    "allow_filter_fallback": ALLOW_FILTER_FALLBACK,
    "generation_configs": GENERATION_CONFIGS,
})


In [ ]:
all_train_records = read_jsonl(TRAIN_FILE)
records_by_id = {str(record.get("id")): record for record in all_train_records}
selected_records = [
    records_by_id[record_id]
    for record_id in SELECTED_DESCRIPTION_IDS
    if record_id in records_by_id
]
missing_ids = [record_id for record_id in SELECTED_DESCRIPTION_IDS if record_id not in records_by_id]
if missing_ids:
    print("Warning: missing description IDs:", missing_ids)
if not selected_records:
    raise ValueError(f"None of the requested description IDs were found: {SELECTED_DESCRIPTION_IDS}")

reference_groups = prepare_reference_groups(all_train_records, METRIC_CONFIG)

selected_preview = [
    {
        "id": str(record.get("id")),
        "description": str(record.get("description")),
        "reference_selfies": str(record.get("selfies")),
        "reference_smiles": str(record.get("source_smiles")),
    }
    for record in selected_records
]
print_records("Selected descriptions", selected_preview)

prompt_variants_by_id = {}
for record in selected_records:
    description_id = str(record.get("id"))
    description = str(record.get("description"))
    prompt_variants_by_id[description_id] = {
        "native_selfies": build_text2mol_prompt(description),
    }

prompt_preview = []
for record in selected_records:
    description_id = str(record.get("id"))
    for prompt_variant, prompt_text in prompt_variants_by_id[description_id].items():
        prompt_preview.append(
            {
                "description_id": description_id,
                "prompt_variant": prompt_variant,
                "prompt_text": prompt_text,
            }
        )
print_records("Prompt preview", prompt_preview)


In [ ]:
class BioT5NativeGenerator:
    # Notebook-local helper that follows the official BioT5 text2mol loading path.

    def __init__(
        self,
        *,
        model_name_or_path: str,
        device_name: str,
        model_max_length: int,
        generation_config: dict[str, object],
    ) -> None:
        self.tokenizer = T5Tokenizer.from_pretrained(
            model_name_or_path,
            model_max_length=model_max_length,
        )
        self.device = choose_device(device_name)
        self.model_max_length = int(model_max_length)
        self.generation_config = dict(generation_config)

        self.model = T5ForConditionalGeneration.from_pretrained(model_name_or_path)
        self.model.to(self.device)
        self.model.eval()
        self.supports_custom_generate = self._supports_custom_generate()

    def _supports_custom_generate(self) -> bool:
        try:
            parameters = inspect.signature(self.model.generate).parameters
        except (TypeError, ValueError):
            return False
        return "custom_generate" in parameters

    def _build_generation_kwargs(self, target_count: int) -> dict[str, object]:
        kwargs: dict[str, object] = {
            "max_length": int(self.generation_config.get("max_length", self.model_max_length)),
            "num_beams": int(self.generation_config.get("num_beams", 1)),
            "num_return_sequences": int(self.generation_config.get("num_return_sequences", target_count)),
            "do_sample": False,
            "use_cache": True,
        }

        if kwargs["num_return_sequences"] != int(target_count):
            raise ValueError(
                f"generation config expected {kwargs['num_return_sequences']} outputs but target_count={int(target_count)}"
            )

        if kwargs["num_beams"] < kwargs["num_return_sequences"]:
            raise ValueError(
                f"num_beams must be >= num_return_sequences, got {kwargs['num_beams']} < {kwargs['num_return_sequences']}"
            )

        if "num_beam_groups" in self.generation_config:
            kwargs["num_beam_groups"] = int(self.generation_config["num_beam_groups"])
            kwargs["diversity_penalty"] = float(self.generation_config["diversity_penalty"])
            if kwargs["num_beam_groups"] <= 1:
                raise ValueError("diverse beam search requires num_beam_groups > 1")
            if kwargs["num_beams"] % kwargs["num_beam_groups"] != 0:
                raise ValueError(
                    f"num_beams must be divisible by num_beam_groups, got {kwargs['num_beams']} and {kwargs['num_beam_groups']}"
                )
            if kwargs["diversity_penalty"] <= 0:
                raise ValueError("diverse beam search requires diversity_penalty > 0")
        if "early_stopping" in self.generation_config:
            kwargs["early_stopping"] = bool(self.generation_config["early_stopping"])
        if "length_penalty" in self.generation_config:
            kwargs["length_penalty"] = float(self.generation_config["length_penalty"])
        return kwargs

    @staticmethod
    def _needs_remote_group_beam_search(exc: Exception) -> bool:
        message = str(exc)
        return (
            "Group Beam Search requires `trust_remote_code=True`" in message
            or "transformers-community/group-beam-search" in message
        )

    @staticmethod
    def _remote_group_beam_generation_kwargs(generation_kwargs: dict[str, object]) -> dict[str, object]:
        remote_kwargs = dict(generation_kwargs)
        remote_kwargs["custom_generate"] = "transformers-community/group-beam-search"
        remote_kwargs["trust_remote_code"] = True
        return remote_kwargs

    def generate_candidates(self, prompt_text: str, target_count: int) -> list[str]:
        encoded = self.tokenizer(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=self.model_max_length,
        )
        encoded = {key: value.to(self.device) for key, value in encoded.items()}
        generation_kwargs = self._build_generation_kwargs(target_count)

        with torch.no_grad():
            try:
                generated_ids = self.model.generate(**encoded, **generation_kwargs)
            except ValueError as exc:
                if not self._needs_remote_group_beam_search(exc):
                    raise
                if not self.supports_custom_generate:
                    raise RuntimeError(
                        "Installed transformers generate() does not expose custom_generate support."
                    ) from exc
                generated_ids = self.model.generate(
                    **encoded,
                    **self._remote_group_beam_generation_kwargs(generation_kwargs),
                )

        raw_texts = self.tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=False,
            clean_up_tokenization_spaces=True,
        )
        outputs = list(raw_texts[: int(target_count)])
        if len(outputs) != int(target_count):
            print(f"Warning: requested {int(target_count)} outputs but decoded {len(outputs)} outputs.")
        return outputs


In [ ]:
first_generation_config_name = next(iter(GENERATION_CONFIGS))
generator = BioT5NativeGenerator(
    model_name_or_path=MODEL_NAME_OR_PATH,
    device_name=DEVICE,
    model_max_length=MODEL_MAX_LENGTH,
    generation_config=GENERATION_CONFIGS[first_generation_config_name],
)

print_json("Generator Preview", {
    "device": str(generator.device),
    "supports_custom_generate": bool(generator.supports_custom_generate),
    "num_selected_descriptions": len(selected_records),
    "generation_config_names": list(GENERATION_CONFIGS.keys()),
})


In [ ]:
raw_generations = []
for generation_config_name, generation_config in GENERATION_CONFIGS.items():
    generator.generation_config = dict(generation_config)
    target_count = int(generation_config.get("target_count", NUM_SAMPLES))
    print_section(f"Running {generation_config_name}")

    for record in selected_records:
        description_id = str(record.get("id"))
        description = str(record.get("description"))
        for prompt_variant, prompt_text in prompt_variants_by_id[description_id].items():
            started_at = time.perf_counter()
            outputs = generator.generate_candidates(prompt_text, target_count)
            elapsed_seconds_batch = time.perf_counter() - started_at
            seconds_per_sample_batch = elapsed_seconds_batch / max(len(outputs), 1)
            print(
                f"{generation_config_name} | {description_id} | {prompt_variant} | "
                f"samples={len(outputs)} | elapsed_seconds={elapsed_seconds_batch:.2f} | "
                f"seconds_per_sample={seconds_per_sample_batch:.3f}"
            )
            for candidate_index, raw_prediction_text in enumerate(outputs):
                raw_generations.append(
                    {
                        "generation_config_name": generation_config_name,
                        "description_id": description_id,
                        "description": description,
                        "prompt_variant": prompt_variant,
                        "candidate_index": candidate_index,
                        "raw_prediction_text": raw_prediction_text,
                        "elapsed_seconds_batch": elapsed_seconds_batch,
                        "seconds_per_sample_batch": seconds_per_sample_batch,
                    }
                )

print_json("Generation Report", {
    "num_rows": len(raw_generations),
    "generation_configs": sorted({row["generation_config_name"] for row in raw_generations}),
    "description_ids": sorted({row["description_id"] for row in raw_generations}),
    "prompt_variants": sorted({row["prompt_variant"] for row in raw_generations}),
})
print_records(
    "Raw generation preview",
    raw_generations,
    limit=12,
    keys=[
        "generation_config_name",
        "description_id",
        "prompt_variant",
        "candidate_index",
        "raw_prediction_text",
    ],
)


In [ ]:
review_rows = []
for row in raw_generations:
    review_rows.append(
        assess_biot5_native_generation_output(
            description_id=row["description_id"],
            description=row["description"],
            prompt_variant=row["prompt_variant"],
            candidate_index=row["candidate_index"],
            raw_prediction_text=row["raw_prediction_text"],
            references=reference_groups[row["description"]],
            metric_config=METRIC_CONFIG,
            generation_config_name=row["generation_config_name"],
            elapsed_seconds_batch=row["elapsed_seconds_batch"],
            seconds_per_sample_batch=row["seconds_per_sample_batch"],
            allow_filter_fallback=ALLOW_FILTER_FALLBACK,
        )
    )

parsed_preview_columns = [
    "generation_config_name",
    "description_id",
    "prompt_variant",
    "candidate_index",
    "raw_prediction_text",
    "cleaned_selfies",
    "parsed_selfies",
    "filtered_selfies",
    "used_filter_selfies_fallback",
    "decoded_smiles",
    "selfies_decode_error",
]
print_records("Raw vs Parsed SELFIES preview", review_rows, limit=20, keys=parsed_preview_columns)

detail_columns = [
    "generation_config_name",
    "description_id",
    "prompt_variant",
    "candidate_index",
    "raw_prediction_text",
    "cleaned_selfies",
    "parsed_selfies",
    "selected_selfies",
    "filtered_selfies",
    "used_filter_selfies_fallback",
    "decoded_smiles",
    "is_valid_selfies",
    "selfies_decode_error",
    "canonical_smiles",
    "derived_selfies",
    "is_valid_smiles",
    "best_reference_smiles",
    "max_dice_similarity",
    "passes_similarity_threshold",
    "rejection_reason",
]
print_records("Assessment preview", review_rows, limit=30, keys=detail_columns)

summary_rows = summarize_biot5_native_review_records(review_rows)
summary_rows = sorted(
    summary_rows,
    key=lambda item: (
        str(item["generation_config_name"]),
        str(item["description_id"]),
        str(item["prompt_variant"]),
    ),
)

summary_columns = [
    "generation_config_name",
    "description_id",
    "prompt_variant",
    "sample_count",
    "elapsed_seconds_batch",
    "seconds_per_sample_batch",
    "valid_selfies_rate",
    "filter_selfies_fallback_rate",
    "filter_selfies_recovery_rate",
    "valid_smiles_rate",
    "unique_canonical_smiles_count",
    "avg_max_dice_similarity",
    "best_max_dice_similarity",
    "passes_similarity_threshold_rate",
    "invalid_selfies_rate",
    "invalid_smiles_rate",
]
print_records("Summary rows", summary_rows, keys=summary_columns)

print_section("Compact Summary")
for summary in summary_rows:
    print(
        f"{summary['generation_config_name']} | {summary['description_id']} | {summary['prompt_variant']} | "
        f"samples={summary['sample_count']} | elapsed={summary['elapsed_seconds_batch']:.2f}s | "
        f"sec_per_sample={summary['seconds_per_sample_batch']:.3f} | "
        f"valid_selfies={summary['valid_selfies_rate']:.3f} | "
        f"filter_recovery={summary['filter_selfies_recovery_rate']:.3f} | "
        f"valid_smiles={summary['valid_smiles_rate']:.3f} | "
        f"unique={summary['unique_canonical_smiles_count']} | "
        f"avg_dice={summary['avg_max_dice_similarity']:.3f} | "
        f"best_dice={summary['best_max_dice_similarity']:.3f}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(RAW_GENERATIONS_PATH, raw_generations)
write_jsonl(REVIEW_ROWS_PATH, review_rows)
SUMMARY_ROWS_PATH.write_text(json_dumps(summary_rows), encoding="utf-8")
CONFIG_SNAPSHOT_PATH.write_text(json_dumps({
    "runtime": runtime_report,
    "hf_login_status": hf_login_status,
    "train_file": str(TRAIN_FILE),
    "model_name_or_path": MODEL_NAME_OR_PATH,
    "selected_description_ids": SELECTED_DESCRIPTION_IDS,
    "num_samples": NUM_SAMPLES,
    "model_max_length": MODEL_MAX_LENGTH,
    "allow_filter_fallback": ALLOW_FILTER_FALLBACK,
    "generation_configs": GENERATION_CONFIGS,
}), encoding="utf-8")

artifact_dir, manifest = export_stage_artifacts(
    stage_name=STAGE_NAME,
    artifact_map={
        "review_outputs": OUTPUT_DIR,
    },
    metadata={
        "runtime": runtime_report,
        "hf_login_status": hf_login_status,
        "model_name_or_path": MODEL_NAME_OR_PATH,
        "selected_description_ids": SELECTED_DESCRIPTION_IDS,
        "num_samples": NUM_SAMPLES,
        "output_dir": str(OUTPUT_DIR),
    },
)

print_json("Saved Outputs", {
    "raw_generations_path": str(RAW_GENERATIONS_PATH),
    "review_rows_path": str(REVIEW_ROWS_PATH),
    "summary_rows_path": str(SUMMARY_ROWS_PATH),
    "config_snapshot_path": str(CONFIG_SNAPSHOT_PATH),
    "artifact_dir": str(artifact_dir),
    "manifest": manifest,
})


## Review Questions

After the notebook runs, use the summary and saved review outputs to answer:

1. Does the native BioT5 path produce decodable SELFIES more reliably than the old custom tokenizer path?
2. How often is `filter_selfies(...)` needed to recover otherwise valid outputs?
3. Does diverse beam improve usable molecule diversity without collapsing validity?
4. Is the native checkpoint strong enough that the old SMILES-tag extraction experiment is no longer needed for this review notebook?
